# Reading one engine of the AGTF30 run-to-failure dataset

This notebook reads one development engine from one subset file with `h5py` and plots its health index and one
measured signal. It needs only `numpy`, `h5py` and `matplotlib`. Set `PATH` (or the environment variable
`AGTF30_FILE`) to a downloaded subset file, for example `AGTF30_ALL.h5`. Each file is 12-14 GB compressed; the code reads only the rows of one engine.

In [ ]:
import os
PATH = os.environ.get("AGTF30_FILE", "AGTF30_ALL.h5")   # any of the eight subset files

In [ ]:
import h5py, numpy as np

with h5py.File(PATH, "r") as f:
    cyc = f["cycles"][:]                      # one row per flight
    dev = cyc[cyc["split"] == b"dev"]         # same order as the *_dev matrices
    end = np.cumsum(dev["seconds"])           # end row of each flight
    sel = np.flatnonzero(dev["unit"] == dev["unit"][0])   # first development engine
    r0, r1 = end[sel[0]] - dev["seconds"][sel[0]], end[sel[-1]]
    names = [n.decode() for n in f["X_s_var"][:]]
    X_s = f["X_s_dev"][r0:r1]                 # noisy measured signals, 1 Hz
    W = f["W_dev"][r0:r1]                     # flight conditions
    rul = f["Y_dev"][r0:r1, 0]                # remaining flights
    hi = dev["HI"][sel]                       # health index after each flight

print(f"engine {dev['unit'][0]}: {len(sel)} flights, {r1 - r0:,} rows, RUL {rul[0]} -> {rul[-1]}")
print("measured signals:", names)

Health index after each flight, and the per-flight mean of the LPT outlet temperature T50 at cruise
(seconds within 5 % of the flight's highest altitude). The cruise mean is a simple feature; the paper's learnability check uses residuals
against a healthy-engine model instead.

In [ ]:
import matplotlib.pyplot as plt

flight = np.repeat(np.arange(1, len(sel) + 1), dev["seconds"][sel])     # flight number of every row
t50 = X_s[:, names.index("T50")]
t50_cruise = []
for k in range(1, len(sel) + 1):
    m = flight == k
    top = W[m, 0] >= 0.95 * W[m, 0].max()          # cruise: near the flight's highest altitude
    t50_cruise.append(t50[m][top].mean())

fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))
ax[0].plot(np.arange(1, len(sel) + 1), hi); ax[0].axhline(0, color="gray", lw=0.8)
ax[0].set(xlabel="flight", ylabel="health index HI")
ax[1].plot(np.arange(1, len(sel) + 1), t50_cruise, ".")
ax[1].set(xlabel="flight", ylabel="T50 at cruise, flight mean (°R)")
plt.tight_layout(); plt.show()